# 🧹 Homework — O quão boa é essa base? 

**Autor:** [Seu Nome]  
**Entidade:** Insper Data — Ciclo Básico  
**Data:** 19 de Fevereiro de 2026

---
### ⚠️ Diagnóstico de Sobrevivência
A limpeza de dados não é um capricho estético; é a diferença entre uma análise que gera lucro e uma que gera prejuízo. O objetivo aqui é simples: **provar com números o quão suja a base está e o quanto conseguimos salvá-la.**

Neste entregável, você deve:
1.  **Diagnosticar:** Encontrar onde a base dói (missing, tipos errados, duplicatas).
2.  **Medir:** Criar um score matemático de qualidade.
3.  **Limpar:** Aplicar as regras mínimas de saneamento.
4.  **Comparar:** Mostrar o ganho de qualidade real através do seu score.

In [3]:
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
import re
from dateutil import parser
from datetime import datetime

# A base vem junto nesta pasta, em data_raw/ — é o original, que fica de referência.
# data/ é a cópia de trabalho, que é a que vamos usar e alterar.
# Bagunçou tudo? Apague data/vendas.csv e rode esta célula de novo.
Path('data').mkdir(exist_ok=True)
if not Path('data/vendas.csv').exists():
    shutil.copy('data_raw/vendas.csv', 'data/vendas.csv')

# --- HELPERS DE CONVERSÃO ---

def to_float_safe(x):
    """Converte string com vírgula/sujeira em float."""
    try: 
        return float(str(x).replace(',', '.'))
    except: 
        return np.nan

def to_int_safe(x):
    """Converte valores numéricos para int (lida com floats de sistema)."""
    try: 
        return int(float(x))
    except: 
        return np.nan

def parse_date_safe(x):
    """Parse de data inteligente (ajuda com diferentes formatos de string)."""
    try: 
        return parser.parse(str(x))
    except: 
        return pd.NaT

# --- HELPERS DE VALIDAÇÃO E LIMPEZA ---

def clean_string(x):
    """Padronização básica: remove espaços e coloca em minúsculo."""
    if pd.isna(x): return np.nan
    return str(x).strip().lower()

def validate_email(email):
    """Valida se o formato do e-mail é aceitável via Regex."""
    if pd.isna(email): return False
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(pattern, str(email)))

def pct(x, total):
    """Gera string de porcentagem formatada para relatórios."""
    if total == 0: return "0 (0.00%)"
    return f"{int(x)} ({100*x/total:.2f}%)"

def get_outliers_iqr(df, column):
    """Detecta outliers usando a regra do Intervalo Interquartil (IQR)."""
    series = df[column].apply(to_float_safe).dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return (series < lower_bound) | (series > upper_bound)


df = pd.read_csv("data/vendas.csv", dtype=object)

---
### 🫵🫵Agora é contigo, use os helpers e responda: Essa base é boa para se usar? Quais métricas são interessantes?
---

In [4]:
df.head(5)

,id_venda,id_cliente,data_venda,categoria,valor_venda,quantidade,desconto,devolvido,pago,forma_pagamento,cidade_entrega,obs
0,6384,452.0,2024-08-11 00:00:00,Alimentos,25.8,3.0,0.0,False,False,NaN,CTBA,telefone inválido
1,4167,188.0,2024-08-09 00:00:00,Informática,1625.53,2.0,0.15,False,True,boleto,SLAVADOR,NaN
2,2077,277.0,2023-08-13 00:00:00,Escritório,232.46,2.0,0.0,False,True,cartao,SAO PAULO CENTRO,NaN
3,4568,964.0,2024-04-20 00:00:00,Eletrônicos,898.51,1.0,0.0,False,True,pix,RJ,telefone inválido
4,3796,551.0,2024-05-30 00:00:00,Escritório,146.13,1.0,0.0,False,True,pix,são paulo,NaN
